In [ ]:
import os
print(os.getcwd())
os.environ["NUPLAN_MAPS_ROOT"] = os.path.expandvars("$HOME/Code/navsim/dataset/maps")
# os.environ["NAVSIM_EXP_ROOT"] = os.path.expandvars("$HOME/Code/navsim/exp")
# os.environ["NAVSIM_DEVKIT_ROOT"] = os.path.expandvars("$HOME/Code/navsim/GTRS")
os.environ["OPENSCENE_DATA_ROOT"] = os.path.expandvars("$HOME/Code/navsim/dataset")
# os.environ["NAVSIM_TRAJPDM_ROOT"] = os.path.expandvars("$HOME/Code/navsim/dataset/traj_pdm_v2")

from pathlib import Path

import hydra
from hydra.utils import instantiate
import matplotlib.pyplot as plt

from navsim.common.dataloader import SceneLoader
from navsim.common.dataclasses import SceneFilter, SensorConfig
from hydra.core.global_hydra import GlobalHydra
SPLIT = "mini"  # ["mini", "test", "trainval"]
FILTER = "all_scenes"
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()
hydra.initialize(config_path="../navsim/planning/script/config/common/train_test_split/scene_filter")
cfg = hydra.compose(config_name=FILTER)
print(cfg)
scene_filter: SceneFilter = instantiate(cfg)
# scene_filter.max_scenes = 8
openscene_data_root = Path(os.getenv("OPENSCENE_DATA_ROOT"))

scene_loader = SceneLoader(
    openscene_data_root / f"navsim_logs/{SPLIT}", # data_path
    openscene_data_root / f"sensor_blobs/{SPLIT}", # original_sensor_path
    scene_filter,
    openscene_data_root / "warmup_two_stage/sensor_blobs", # synthetic_sensor_path
    openscene_data_root / "warmup_two_stage/synthetic_scene_pickles", # synthetic_scenes_path
    sensor_config=SensorConfig.build_all_sensors(),
)

In [ ]:
from diffusion_planner import DiffusionPlanner, cfg
from diffusion_planner_dataset import DiffusionPlannerDataset
from torch.utils.data.dataloader import DataLoader

dataset = DiffusionPlannerDataset(
    scene_loader=scene_loader,
    cfg = cfg,
    max_len = 8
)
data_loader = DataLoader(dataset=dataset, batch_size=1, shuffle=False)

In [ ]:
import torch
from diffusion_planner import ObservationNormalizer

model = DiffusionPlanner(cfg)
ckpt_path = '/Users/chenran/Code/diffusion-planner/ckpt/model.pth'
ckpt = torch.load(ckpt_path, map_location='cpu')
state_dict = ckpt['ema_state_dict']
state_dict = {
    k.replace('module.', '', 1): v
    for k, v in state_dict.items()
}
model.load_state_dict(state_dict, strict=True)
model.eval()

observation_normalizer = ObservationNormalizer({
    k: {kk: torch.tensor(vv, dtype=torch.float32) for kk, vv in v.items()}
    for k, v in cfg['observation_normalizer'].items()
})

for idx, (token, features, targets) in enumerate(data_loader):
    features['ego_current_state'] = torch.tensor(
        [[0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]], dtype=torch.float32
    ).expand_as(features['ego_current_state']).clone()
    features_n = observation_normalizer(features)
    op = model(features_n)
    if idx > 1:
        break

from visualization import show_diffusion_planner_result
show_diffusion_planner_result(op, targets, features)

In [ ]:
import torch.nn as nn
from typing import Any, Callable
from diffusion_planner import StateNormalizer
def diffusion_loss_func(
        model: nn.Module,
        inputs: dict[str, torch.Tensor],
        marginal_prob: Callable[[torch.Tensor], torch.Tensor],
        futures: tuple[torch.Tensor, torch.Tensor],
        norm: StateNormalizer,
        loss: dict[str, Any],
        model_type: str,
        eps: float=1e-3,
):
    ego_future, neighbors_future, neighbor_future_mask = futures
    neighbors_future_valid = ~neighbor_future_mask

    B, Pn, T, _ = neighbors_future.shape
    ego_current, neighbors_current = inputs['ego_current_state'][:, :4], inputs['neighbor_agents_past'][:, :Pn, -1, :4]
    neighbor_current_mask = torch.sum(torch.ne(neighbors_current[..., :4], 0), dim=-1) == 0
    neighbor_mask = torch.concat((neighbor_current_mask.unsqueeze(-1), neighbor_future_mask), dim=-1)
    
    gt_future = torch.cat([ego_future[:, None, :, :], neighbors_future[..., :]], dim=1)
    current_states = torch.cat([ego_current[:, None], neighbors_current], dim=1)

    P = gt_future.shape[1]
    t = torch.rand(B, device=gt_future.device) * (1 - eps) + eps
    z = torch.randn_like(gt_future, device=gt_future.device)

    all_gt = torch.cat([current_states[:, :, None, :], norm(gt_future)], dim=2)
    all_gt[:, 1:][neighbor_mask] = 0.0

    mean, std = marginal_prob(all_gt[..., 1:, :], t)
    std = std.view(-1, *([1] * (len(all_gt[..., 1:, :].shape)-1)))

    xT = mean + std * z
    xT = torch.cat([all_gt[:, :, :1, :], xT], dim=2)

    merged_inputs = {
        **inputs,
        'sampled_trajectories': xT,
        'diffusion_time': t,
    }

    _, decoder_output = model(merged_inputs)
    score = decoder_output['score'][:, :, 1:, :]

    if model_type == 'score':
        dpm_loss = torch.sum((score * std + z)**2, dim=-1)
    elif model_type == 'x_start':
        dpm_loss = torch.sum((score-all_gt[:, :, 1:, :])**2, dim=-1)
    
    masked_prediction_loss = dpm_loss[:, 1:, :][neighbors_future_valid]

    if masked_prediction_loss.numel() > 0:
        loss['neighbor_prediction_loss'] = masked_prediction_loss.mean()
    else:
        loss['neighbor_prediction_loss'] = torch.tensor(0.0, device=masked_prediction_loss.device)

    loss['ego_planning_loss'] = dpm_loss[:, 0, :].mean()

    assert not torch.isnan(dpm_loss).sum(), f'loss cannot be nan, z={z}'

    return loss, decoder_output

In [28]:
import torch
from torch import optim
from diffusion_planner import ObservationNormalizer
from tqdm import tqdm

model = DiffusionPlanner(cfg)
optimizer = optim.AdamW([{'params': model.parameters(), 'lr': 0.001}])
device = 'cpu'
observation_normalizer = ObservationNormalizer({
    k: {kk: torch.tensor(vv, dtype=torch.float32) for kk, vv in v.items()}
    for k, v in cfg['observation_normalizer'].items()
})
state_normalizer = StateNormalizer(**cfg['state_normalizer'])
for epoch in range(0, 1):
    with tqdm(data_loader, desc='Training', unit='batch') as data_epoch:
        for token, features, targets in data_epoch:
            print(targets.keys())
            for k, v in features.items():
                v = v.to(device)
            for k, v in targets.items():
                v = v.to(device)
            
            ego_future = targets['ego_future_gt'].to(device)
            neighbors_future = targets['neighbors_future_gt'].to(device)
            mask = targets['neighbor_future_mask']
            neighbors_future[mask] = 0
            inputs = observation_normalizer(features)
            optimizer.zero_grad()
            loss = {}

            loss, _ = diffusion_loss_func(
                model,
                inputs,
                model.sde.marginal_prob,
                (ego_future, neighbors_future, mask),
                state_normalizer,
                loss,
                cfg['diffusion_model_type']
            )
            
            loss['loss'] = loss['neighbor_prediction_loss'] + 0.1 * loss['ego_planning_loss']

            total_loss = loss['loss'].item()

            loss['loss'].backward()

            nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()
            


Training:  12%|█▎        | 1/8 [00:00<00:01,  4.06batch/s]

dict_keys(['ego_future_gt', 'neighbors_future_gt', 'neighbor_future_mask', 'trajectory'])


Training:  25%|██▌       | 2/8 [00:00<00:01,  4.88batch/s]

dict_keys(['ego_future_gt', 'neighbors_future_gt', 'neighbor_future_mask', 'trajectory'])
dict_keys(['ego_future_gt', 'neighbors_future_gt', 'neighbor_future_mask', 'trajectory'])


Training:  50%|█████     | 4/8 [00:00<00:00,  5.23batch/s]

dict_keys(['ego_future_gt', 'neighbors_future_gt', 'neighbor_future_mask', 'trajectory'])
dict_keys(['ego_future_gt', 'neighbors_future_gt', 'neighbor_future_mask', 'trajectory'])


Training:  75%|███████▌  | 6/8 [00:01<00:00,  5.14batch/s]

dict_keys(['ego_future_gt', 'neighbors_future_gt', 'neighbor_future_mask', 'trajectory'])
dict_keys(['ego_future_gt', 'neighbors_future_gt', 'neighbor_future_mask', 'trajectory'])


Training: 100%|██████████| 8/8 [00:01<00:00,  5.15batch/s]

dict_keys(['ego_future_gt', 'neighbors_future_gt', 'neighbor_future_mask', 'trajectory'])
